## Notebook Instructions

1. If you are new to Jupyter notebooks, please go through this introductory manual [here](https://quantra.quantinsti.com/quantra-notebook).
2. Any changes made in this notebook would be lost after you close the browser window. **You can download the notebook to save your work on your PC.**
3. Before running this notebook on your local PC:
   - You need to set up a Python environment and the relevant packages on your local PC. To do so, go through the section on **"Run Codes Locally on Your Machine"** in the course.
   - You need to **download the zip file available in the last unit** of this course. The zip file contains the data files and/or Python modules that might be required to run this notebook.


# Fetching Full Instrument Lists

Every trading platform has a master list of all tradable instruments, from stocks and indices to commodities and derivatives. The KiteConnect API provides access to this entire list through a single API call. This master file is crucial for finding the correct `instrument_token` for placing orders, fetching historical data, or subscribing to live market data.

This notebook will guide you through:
1.  **Downloading the complete instrument list** using an authenticated `KiteConnect` object.
2.  **Processing the data** by loading it into a Pandas DataFrame for efficient handling.
3.  **Filtering instruments** by exchange segments (like NSE, BSE, NFO, MCX) to find exactly what you need.

## Prerequisites

* **An Authenticated `kite` Object**: This notebook assumes you have already authenticated.
* **Pandas Library**: We will use the Pandas library to work with the instrument data.

## Creating the `KiteAuth` Class

The following code defines a `KiteAuth` class that facilitates the manual authentication process with the Zerodha Kite Connect API. It prompts the user to log in via a URL, captures the request token, and generates an access token required for API access. Once authenticated, it returns a ready-to-use KiteConnect object for placing orders, fetching market data, and more.

In [ ]:
# Import the KiteConnect class from the kiteconnect package to interact with Zerodha's Kite API
from kiteconnect import KiteConnect


class KiteAuth:
    """A class to handle the KiteConnect authentication flow."""
    def __init__(self, api_key, api_secret):
        """
        Initialises the KiteAuth object with API credentials.

        Parameters:
        - api_key: Your Zerodha Kite API key
        - api_secret: Your Zerodha Kite API secret
        """
        self.api_key = api_key  # Store the API key
        self.api_secret = api_secret  # Store the API secret
        self.kite = None  # Will hold the authenticated KiteConnect object

    def authenticate(self):
        """
        Guides the user through the manual authentication process.

        Steps:
        - Initialises KiteConnect
        - Displays the login URL to the user
        - Prompts the user to input the request token received after login
        - Uses the token to generate a session and retrieve the access token
        - Sets the access token to the KiteConnect instance
        - Returns the authenticated KiteConnect object if successful
        """
        try:
            # Create a KiteConnect instance with the given API key
            kite = KiteConnect(api_key=self.api_key)

            # Instruct the user to log in via the URL to generate a request token
            print("Please click on the URL below to login:")
            print(kite.login_url())
            print("=" * 50)

            # Accept the request token manually from the user
            request_token = input(
                "Enter the request token from redirect URL: ").strip()

            # Generate a session using the request token and API secret
            data = kite.generate_session(request_token,
                                         api_secret=self.api_secret)

            # Set the access token in the KiteConnect instance for future API calls
            kite.set_access_token(data['access_token'])

            # Save the authenticated KiteConnect instance
            self.kite = kite

            # Print a welcome message with the user's name
            print("=" * 50)
            print("Authentication successful!")
            print(f"Welcome!")
            print("=" * 50)

            return kite  # Return the authenticated KiteConnect instance

        except Exception as e:
            # Handle errors that may occur during authentication
            print(f"Authentication failed: {e}")
            return None

## Authenticating and Fetching Instruments

Input your api key and api secret in the place holders `YOUR_API_KEY` and `YOUR_API_SECRET` in the below code to authenticate your login.

In [ ]:
import pandas as pd

# --- Start Authentication ---
# IMPORTANT: Replace with your credentials
# Fetch API credentials securely from .env file
API_KEY = "YOUR_API_KEY"
API_SECRET = "YOUR_API_SECRET"

kite = None
if API_KEY != "YOUR_API_KEY":
    # This is how you would use the imported class
    auth = KiteAuth(API_KEY, API_SECRET)
    kite = auth.authenticate()
else:
    print(
        "\nPlease update your API_KEY and API_SECRET to run the authentication."
    )

Please click on the URL below to login:
https://kite.zerodha.com/connect/login?api_key=tj19jzf6vgdl162v&v=3
Enter the request token from redirect URL: OVrOhbmUC9ctP7QWrMYtnjtRlDE1cUtk
Authentication successful!
Welcome!


## Download and Process the Instrument Master

With our authenticated `kite` object, we can now download the full list of instruments. The API returns a list of dictionaries, which is perfect for loading into a Pandas DataFrame.

In [ ]:
if kite:
    try:
        # Fetch the complete instrument list
        instruments = kite.instruments()

        # Convert the list of dictionaries to a Pandas DataFrame
        instrument_df = pd.DataFrame(instruments)

        print(
            f"Successfully downloaded and processed {len(instrument_df)} instruments."
        )

        # Display the first few rows and the columns of the DataFrame
        print("\nDataFrame Columns:")
        print(instrument_df.columns)

        print("\nDataFrame Head:")
        display(instrument_df.head())

    except Exception as e:
        print(f"Error fetching or processing instruments: {e}")
else:
    print("Authentication was not successful. Cannot fetch instruments.")

Successfully downloaded and processed 94200 instruments.

DataFrame Columns:
Index(['instrument_token', 'exchange_token', 'tradingsymbol', 'name',
       'last_price', 'expiry', 'strike', 'tick_size', 'lot_size',
       'instrument_type', 'segment', 'exchange'],
      dtype='object')

DataFrame Head:


,instrument_token,exchange_token,tradingsymbol,name,last_price,expiry,strike,tick_size,lot_size,instrument_type,segment,exchange
0,281854981,1100996,BANKEX25JUNFUT,BANKEX,0.0,2025-06-24,0.0,0.05,30,FUT,BFO-FUT,BFO
1,292126213,1141118,BANKEX25JULFUT,BANKEX,0.0,2025-07-29,0.0,0.05,30,FUT,BFO-FUT,BFO
2,211672581,826846,BANKEX25AUGFUT,BANKEX,0.0,2025-08-26,0.0,0.05,30,FUT,BFO-FUT,BFO
3,281836549,1100924,SENSEX25JUNFUT,SENSEX,0.0,2025-06-24,0.0,0.05,20,FUT,BFO-FUT,BFO
4,210402821,821886,SENSEX25701FUT,SENSEX,0.0,2025-07-01,0.0,0.05,20,FUT,BFO-FUT,BFO


## Filtering Instruments by Exchange

The `instrument_df` DataFrame contains instruments from all exchanges. The `exchange` column is key to filtering this data. Let's see what unique exchanges are available.

In [ ]:
if 'instrument_df' in locals():

    # Find all unique values in the 'exchange' column
    available_exchanges = instrument_df['exchange'].unique()
    print("Available Exchanges:", available_exchanges)

Available Exchanges: ['BFO' 'BSE' 'CDS' 'NSE' 'MCX' 'NSEIX' 'GLOBAL' 'NCO' 'NFO']


### Example 1: Filter for NSE Equities

To get a list of all stocks trading on the National Stock Exchange (NSE), we filter where the `exchange` column is 'NSE'.

In [ ]:
if 'instrument_df' in locals():
    nse_stocks = instrument_df[instrument_df['exchange'] == 'NSE']

    print(f"Found {len(nse_stocks)} instruments in the NSE segment.")
    print("\nExample NSE stocks:")

    # We also filter by segment to ensure we only get equities
    display(nse_stocks[nse_stocks['segment'] == 'NSE'].head())

Found 7957 instruments in the NSE segment.

Example NSE stocks:


,instrument_token,exchange_token,tradingsymbol,name,last_price,expiry,strike,tick_size,lot_size,instrument_type,segment,exchange
86360,769,3,GOLDSTAR-ST,GOLDSTAR POWER,0.0,,0.0,0.05,11250,EQ,NSE,NSE
86361,1025,4,21STCENMGM,21ST CENTURY MGMT SERVICE,0.0,,0.0,0.01,1,EQ,NSE,NSE
86362,1793,7,AARTIIND,AARTI INDUSTRIES,0.0,,0.0,0.05,1,EQ,NSE,NSE
86363,2561,10,ABAN,ABAN OFFSHORE,0.0,,0.0,0.01,1,EQ,NSE,NSE
86364,3329,13,ABB,ABB INDIA,0.0,,0.0,0.50,1,EQ,NSE,NSE


### Example 2: Filter for NFO (Futures & Options)

To find derivatives, we filter for the 'NFO' exchange. This segment includes stock options, stock futures, index options, and index futures.

In [ ]:
if 'instrument_df' in locals():
    nfo_instruments = instrument_df[instrument_df['exchange'] == 'NFO']

    print(f"Found {len(nfo_instruments)} instruments in the NFO segment.")
    print("\nExample NFO instruments (e.g., NIFTY options):")

    # Let's find a specific NIFTY option
    nifty_options = nfo_instruments[
        (nfo_instruments['name'] == 'NIFTY')
        & (nfo_instruments['instrument_type'] == 'CE')  # Call options
    ]
    display(nifty_options.sort_values(by='expiry').head())

Found 35723 instruments in the NFO segment.

Example NFO instruments (e.g., NIFTY options):


,instrument_token,exchange_token,tradingsymbol,name,last_price,expiry,strike,tick_size,lot_size,instrument_type,segment,exchange
51302,13056258,51001,NIFTY2561924800CE,NIFTY,0.0,2025-06-19,24800.0,0.05,75,CE,NFO-OPT,NFO
51422,13043458,50951,NIFTY2561923900CE,NIFTY,0.0,2025-06-19,23900.0,0.05,75,CE,NFO-OPT,NFO
51420,13043970,50953,NIFTY2561923950CE,NIFTY,0.0,2025-06-19,23950.0,0.05,75,CE,NFO-OPT,NFO
51418,13044482,50955,NIFTY2561924000CE,NIFTY,0.0,2025-06-19,24000.0,0.05,75,CE,NFO-OPT,NFO
51416,13045506,50959,NIFTY2561924050CE,NIFTY,0.0,2025-06-19,24050.0,0.05,75,CE,NFO-OPT,NFO


### Example 3: Filter for MCX (Commodities)

For commodities like Gold, Silver, and Crude Oil, we filter for the 'MCX' exchange.

In [ ]:
if 'instrument_df' in locals():
    mcx_instruments = instrument_df[instrument_df['exchange'] == 'MCX']

    print(f"Found {len(mcx_instruments)} instruments in the MCX segment.")
    print("\nExample MCX instruments (e.g., GOLD futures):")

    # Let's find Gold futures
    gold_futures = mcx_instruments[
        (mcx_instruments['name'] == 'GOLD')
        & (mcx_instruments['segment'] == 'MCX-FUT')  # Futures segment
    ]
    display(gold_futures.sort_values(by='expiry').head())

Found 14421 instruments in the MCX segment.

Example MCX instruments (e.g., GOLD futures):


,instrument_token,exchange_token,tradingsymbol,name,last_price,expiry,strike,tick_size,lot_size,instrument_type,segment,exchange
29926,112236807,438425,GOLD25AUGFUT,GOLD,0.0,2025-08-05,0.0,1.0,1,FUT,MCX-FUT,MCX
29927,112880391,440939,GOLD25OCTFUT,GOLD,0.0,2025-10-03,0.0,1.0,1,FUT,MCX-FUT,MCX
29928,113920775,445003,GOLD25DECFUT,GOLD,0.0,2025-12-05,0.0,1.0,1,FUT,MCX-FUT,MCX
29929,115080711,449534,GOLD26FEBFUT,GOLD,0.0,2026-02-05,0.0,1.0,1,FUT,MCX-FUT,MCX
29930,116433415,454818,GOLD26APRFUT,GOLD,0.0,2026-04-02,0.0,1.0,1,FUT,MCX-FUT,MCX


## Conclusion

You now have a powerful and repeatable method for downloading and filtering the entire universe of instruments available on Zerodha. By converting the instrument list into a `Pandas` DataFrame, you can easily query it to find any instrument you need based on its symbol, exchange, expiry date, or instrument type. This is a fundamental building block for almost any automated trading strategy or analysis tool you will build. <br><br>